In [1]:
import gzip
import io
from datetime import datetime, timedelta
from typing import Any

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MEDIAN_QUANTILE = 0.5

def _format_location_name(location: str) -> str:
    """Convert location name to clean readable name."""
    return location.replace("_", " ")


def plot_quantiles(
    df_quantiles: pd.DataFrame,
    value_col: str,
    date_col: str = "date",
    quantile_col: str = "quantile",
    color: str = "C0",
    title: str | None = None,
    ax: plt.Axes | None = None,
    marker: str = 'o',
    zorder: float = 2,
    label: str | None = None,
) -> tuple[plt.Figure | None, plt.Axes, list, list]:
    """Plot quantile ribbons from quantile DataFrame. Returns legend handles and labels."""
    if df_quantiles.empty:
        msg = "Quantile data is empty"
        raise ValueError(msg)

    required_cols = [date_col, quantile_col, value_col]
    missing = set(required_cols) - set(df_quantiles.columns)
    if missing:
        msg = f"Missing columns: {missing}"
        raise ValueError(msg)

    df_quantile = df_quantiles.pivot(index=date_col, columns=quantile_col, values=value_col)
    df_quantile.index = pd.to_datetime(df_quantile.index)

    fig = None
    if ax is None:
        fig, ax = plt.subplots()

    quantiles = sorted(df_quantile.columns.tolist())
    upper = 0.975
    lower = 0.025
    # Plot 95% CI
    ax.fill_between(
        df_quantile.index,
        df_quantile[upper].values,
        df_quantile[lower].values,
        color=color,
        alpha=0.2,
        linewidth=0,
        zorder=zorder,
    )

    # Plot IQR
    if 0.25 in df_quantile.columns and 0.75 in df_quantile.columns:
        ax.fill_between(
            df_quantile.index,
            df_quantile[0.25].values,
            df_quantile[0.75].values,
            color=color,
            alpha=0.4,
            linewidth=0,
            zorder=zorder + 0.1,
        )

    # Plot median
    if MEDIAN_QUANTILE in df_quantile.columns:
        if marker is not None:
            ax.plot(
                df_quantile.index,
                df_quantile[MEDIAN_QUANTILE].values,
                color=color,
                linewidth=1.5,
                marker=marker,
                markersize=8,
                zorder=zorder + 0.2,
                label=label,
            )
        else:
            ax.plot(
                df_quantile.index,
                df_quantile[MEDIAN_QUANTILE].values,
                color=color,
                linewidth=1.5,
                zorder=zorder + 0.2,
                label=label,
            )

    for tick in ax.get_xticklabels():
        tick.set_rotation(45)
        tick.set_ha("right")

    if title is not None:
        ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.grid(visible=True, linestyle="--", alpha=0.3, linewidth=0.5)

    # Create legend handles for this forecast
    legend_handles = [mpatches.Patch(color=color, alpha=0.4)]
    legend_labels = [label if label else "Forecast"]

    return fig, ax, legend_handles, legend_labels


def plot_surveillance_scatter(
    df_surveillance: pd.DataFrame,
    date_col: str = "date",
    value_col: str = "value",
    color: str = "black",
    size: float = 20.0,
    title: str | None = None,
    ax: plt.Axes | None = None,
    zorder: float = 10,
) -> tuple[plt.Figure | None, plt.Axes]:
    """Plot surveillance data as scatter points."""
    if df_surveillance.empty:
        msg = "Surveillance data is empty"
        raise ValueError(msg)

    missing = {date_col, value_col} - set(df_surveillance.columns)
    if missing:
        msg = f"Missing columns: {missing}"
        raise ValueError(msg)

    df_surveillance = df_surveillance.copy()
    df_surveillance[date_col] = pd.to_datetime(df_surveillance[date_col])
    df_surveillance = df_surveillance.sort_values(date_col)

    fig = None
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 4))

    ax.scatter(df_surveillance[date_col], df_surveillance[value_col], color=color, s=size, zorder=zorder)
    ax.set_xlabel("")

    if title is not None:
        ax.set_title(title)

    for tick in ax.get_xticklabels():
        tick.set_rotation(45)
        tick.set_ha("right")

    return fig, ax


def plot_multi_forecast_grid(
    all_location_quantiles: list[dict[str, pd.DataFrame]],
    forecast_labels: list[str],
    forecast_colors: list[str],
    value_col: str,
    date_col: str = "date",
    quantile_col: str = "quantile",
    panels_per_row: int = 4,
    figsize: tuple[float, float] | None = None,
    location_surveillance: dict[str, pd.DataFrame] | None = None,
    surveillance_date_col: str = "date",
    surveillance_value_col: str = "value",
    surveillance_size: float = 25.0,
) -> tuple[plt.Figure, np.ndarray]:
    """Create grid of quantile plots with multiple forecasts per panel."""
    
    # Get all locations from first forecast
    locations = list(all_location_quantiles[0].keys())
    n = len(locations)
    ncols = panels_per_row
    nrows = int(np.ceil(n / ncols))

    if figsize is None:
        figsize = (4 * ncols, 3.6 * nrows)

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)

    for idx, location in enumerate(locations):
        r = idx // ncols
        c = idx % ncols
        ax = axes[r, c]

        # Plot each forecast with its color
        for forecast_idx, (loc_quantiles, label, color) in enumerate(
            zip(all_location_quantiles, forecast_labels, forecast_colors)
        ):
            if location in loc_quantiles:
                plot_quantiles(
                    df_quantiles=loc_quantiles[location],
                    value_col=value_col,
                    date_col=date_col,
                    quantile_col=quantile_col,
                    color=color,
                    ax=ax,
                    marker='o',
                    zorder=2 + forecast_idx * 0.5,
                    label=label,
                )

        # Add surveillance overlay if provided
        if location_surveillance is not None and location in location_surveillance:
            plot_surveillance_scatter(
                df_surveillance=location_surveillance[location],
                date_col=surveillance_date_col,
                value_col=surveillance_value_col,
                size=surveillance_size,
                ax=ax,
                zorder=10,
            )

        ax.set_title(_format_location_name(location), fontsize=10)

        # Add legend only to first panel in each row
        if idx % panels_per_row == 0:
            # Create legend handles for all forecasts (lines with markers)
            legend_handles = [plt.Line2D([0], [0], color=c, marker='o', markersize=4, 
                                        linewidth=1.5, label=l) 
                            for l, c in zip(forecast_labels, forecast_colors)]
            # Add observed points
            legend_handles.append(plt.Line2D([0], [0], color='black', marker='o', 
                                            linestyle='None', markersize=5, label='Observed'))
            # Add single grey box for uncertainty interval
            lower = 0.025
            upper = 0.975
            legend_handles.append(mpatches.Patch(color='grey', alpha=0.3, 
                                                label=f'{(upper-lower)*100}% CrI'))
            ax.legend(handles=legend_handles, loc='upper left', fontsize=6)

    # Remove unused axes
    for idx in range(len(locations), nrows * ncols):
        r = idx // ncols
        c = idx % ncols
        axes[r, c].axis("off")

    plt.tight_layout()

    return fig, axes


In [69]:
surv_df['location'].values

array(['28', '30', '31', ..., '25', '26', '27'],
      shape=(8517,), dtype=object)

In [75]:
forecast = pd.read_csv('pipeline_flu_202550_smc_rmse_202543-202549_20251210-170328-0cb835c3_outputs_20251210-204958_output_hub_formatted.csv.gz')
forecast['location'].unique()

array(['US', '01', '02', '04', '05', '06', '08', '09', '10', '11', '12',
       '13', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24',
       '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35',
       '36', '37', '38', '39', '40', '41', '42', '44', '45', '46', '47',
       '48', '49', '50', '51', '53', '54', '55', '56'], dtype=object)

In [99]:
# ============================================================================
# Configuration - UPDATE THESE PATHS AND LABELS FOR YOUR FILES
# ============================================================================
import cmocean

def get_colors_from_cmap(n, cmap=cmocean.cm.deep):
    """Get n evenly spaced colors from a colormap."""
    return [cmap(x) for x in np.linspace(0.25, 0.8, n)]

# Example usage:
colors = get_colors_from_cmap(4)

# List of (file_path, label, color) tuples for each forecast file
FORECAST_FILES = [
    ('pipeline_flu_202551_smc_rmse_202544-202550_20251217-172737-d9252dfa_outputs_20251217-184737_output_hub_formatted.csv.gz',
    'smc-rmse', colors[0]),
    # ('pipeline_flu_202550_smc_rmse_202543-202549_augmented_20251210-170500-cd40775b_outputs_20251210-232749_output_hub_formatted.csv.gz',
    # 'smc-rmse', colors[0]),
    # ('pipeline_flu_202550_smc_wmape_202543-202549_augmented_20251210-170742-87d8536e_outputs_20251210-232756_output_hub_formatted.csv.gz', 
    # 'smc-wmape', colors[1]),
    # ('pipeline_flu_202550_top-fraction_rmse_202543-202549_augmented_20251210-171037-9341de32_outputs_20251210-232812_output_hub_formatted.csv.gz', 
    # 'topfraction-rmse', colors[2]),
    # ('pipeline_flu_202550_top-fraction_wmape_202543-202549_augmented_20251210-171151-ed5d1b03_outputs_20251210-232752_output_hub_formatted.csv.gz', 
    # 'topfraction-wmape', colors[3]),
]

SURVEILLANCE_FILE = 'dfed.csv'
TARGET_NAME = 'wk inc flu prop ed visits'
WEEKS_BEFORE_FORECAST = 10

# ============================================================================
# Data loading and preparation
# ============================================================================

# Load surveillance data
surv_df = pd.read_csv(SURVEILLANCE_FILE)

# Map FIPS codes to state names
fips_to_name = surv_df.drop_duplicates('location').set_index('location')['location_name'].to_dict()

# Get unique locations (states, excluding US)
locations = [loc for loc in surv_df['location'].unique() if loc != 'US']
print(locations)

# Load all forecast files
all_location_quantiles = []
forecast_labels = []
forecast_colors = []

for file_path, label, color in FORECAST_FILES:
    print(f"Loading {label} from {file_path}...")
    
    # Load forecast data
    if file_path.endswith('.gz'):
        with gzip.open(file_path, 'rt') as f:
            forecast_df = pd.read_csv(f)
    else:
        forecast_df = pd.read_csv(file_path)
    
    # Filter to target
    forecast_df = forecast_df[forecast_df['target'] == TARGET_NAME].copy()
    forecast_df['location'] = forecast_df['location'].astype(str)
    print(forecast_df['location'].unique())
    # Prepare quantiles for each location
    location_quantiles = {}
    for loc in sorted(locations):
        forecast_loc = forecast_df[forecast_df['location'] == loc].copy()
        if not forecast_loc.empty:
            quantile_df = forecast_loc[['target_end_date', 'output_type_id', 'value']].copy()
            quantile_df.columns = ['date', 'quantile', 'value']
            quantile_df['quantile'] = quantile_df['quantile'].astype(float)
            location_quantiles[fips_to_name.get(loc, loc)] = quantile_df
            print(quantile_df)
    
    all_location_quantiles.append(location_quantiles)
    forecast_labels.append(label)
    forecast_colors.append(color)

# Prepare surveillance data for each location
location_surveillance = {}
for loc in sorted(locations):
    surv_loc = surv_df[surv_df['location'] == loc].copy()
    if not surv_loc.empty:
        surv_formatted = surv_loc[['date', 'value']].copy()
        location_surveillance[fips_to_name.get(loc, loc)] = surv_formatted


# ============================================================================
# Filter surveillance to 8 weeks before forecast through end of forecast
# ============================================================================

for loc in location_surveillance:
    # Get date range across all forecasts
    all_forecast_dates = []
    for loc_quantiles in all_location_quantiles:
        if loc in loc_quantiles:
            all_forecast_dates.extend(pd.to_datetime(loc_quantiles[loc]['date']).tolist())
    
    if all_forecast_dates:
        min_forecast_date = min(all_forecast_dates)
        max_forecast_date = max(all_forecast_dates)
        
        # Start 8 weeks before the forecast period
        start_date = min_forecast_date - timedelta(weeks=WEEKS_BEFORE_FORECAST)
        
        df = location_surveillance[loc]
        df['date'] = pd.to_datetime(df['date'])
        location_surveillance[loc] = df[
            (df['date'] >= start_date) & (df['date'] <= max_forecast_date)
        ]


# ============================================================================
# Create plot
# ============================================================================

print(forecast_df.groupby(['location', 'target']).size())

fig, axes = plot_multi_forecast_grid(
    all_location_quantiles=all_location_quantiles,
    forecast_labels=forecast_labels,
    forecast_colors=forecast_colors,
    value_col='value',
    date_col='date',
    quantile_col='quantile',
    panels_per_row=4,
    location_surveillance=location_surveillance,
    surveillance_date_col='date',
    surveillance_value_col='value',
    surveillance_size=10.0,
)

fig.suptitle('Flu Prop ED Visits: Forecasts vs Surveillance', fontsize=14, y=1.01)
fig.supylabel('Proportion ED Visits', fontsize=11)

plt.savefig('flu_prop_ed_smc_rmse.pdf', dpi=150, bbox_inches='tight')
plt.close()

['28', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '44', '45', '46', '47', '56', '01', '02', '04', '05', '06', '08', '09', '10', '11', '12', '13', '15', '16', '17', '18', '19', '48', '49', '50', '51', '53', '54', '55', '20', '21', '22', '23', '24', '25', '26', '27']
Loading smc-rmse from pipeline_flu_202551_smc_rmse_202544-202550_20251217-172737-d9252dfa_outputs_20251217-184737_output_hub_formatted.csv.gz...
['US' '01' '02' '04' '05' '06' '08' '09' '10' '11' '12' '13' '15' '16'
 '17' '18' '19' '20' '21' '22' '23' '24' '25' '26' '27' '28' '29' '30'
 '31' '32' '33' '34' '35' '36' '37' '38' '39' '40' '41' '42' '44' '45'
 '46' '47' '48' '49' '50' '51' '53' '54' '55' '56']
            date  quantile     value
6095  2025-12-13      0.01  0.015907
6096  2025-12-20      0.01  0.022092
6097  2025-12-27      0.01  0.027100
6098  2026-01-03      0.01  0.035053
6099  2026-01-10      0.01  0.043890
...          ...       ...       ...
6205  2025-12-13      0.99  0.

### Side by side

In [ ]:
dfed = pd.read_csv('https://raw.githubusercontent.com/cdcepi/FluSight-forecast-hub/refs/heads/main/target-data/target-ed-visits-prop.csv')


In [5]:
# ============================================================================
# Configuration - UPDATE THESE PATHS AND LABELS FOR YOUR FILES
# ============================================================================
import cmocean
import gzip
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta

def get_colors_from_cmap(n, cmap=cmocean.cm.deep):
    """Get n evenly spaced colors from a colormap."""
    return [cmap(x) for x in np.linspace(0.25, 0.8, n)]

# Example usage:
colors = get_colors_from_cmap(4)

# List of (file_path, label, color) tuples for each forecast file
FORECAST_FILES = [
    ('pipeline_flu_202551_smc_wmape_202544-202550_augmented_20251217-191414-43dc0c2d_outputs_20251217-203202_output_hub_formatted.csv.gz',
    'smc_wmape_augmented', colors[0]),
]

SURVEILLANCE_FILE = 'dfed.csv'
TARGET_NAME = 'wk inc flu prop ed visits'

# Configuration for the two side-by-side panels
WEEKS_BEFORE_FORECAST_LEFT = 60   # Left panel: more historical data
WEEKS_BEFORE_FORECAST_RIGHT = 8   # Right panel: less historical data

# ============================================================================
# Data loading and preparation
# ============================================================================

# Load surveillance data
surv_df = pd.read_csv(SURVEILLANCE_FILE)

# Map FIPS codes to state names
fips_to_name = surv_df.drop_duplicates('location').set_index('location')['location_name'].to_dict()

# Get unique locations (states, excluding US)
locations = [loc for loc in surv_df['location'].unique() if loc != 'US']

# Load all forecast files
all_location_quantiles = []
forecast_labels = []
forecast_colors = []

for file_path, label, color in FORECAST_FILES:
    print(f"Loading {label} from {file_path}...")
    
    # Load forecast data
    if file_path.endswith('.gz'):
        with gzip.open(file_path, 'rt') as f:
            forecast_df = pd.read_csv(f)
    else:
        forecast_df = pd.read_csv(file_path)
    
    # Filter to target
    forecast_df = forecast_df[forecast_df['target'] == TARGET_NAME].copy()
    forecast_df['location'] = forecast_df['location'].astype(str)
    
    # Prepare quantiles for each location
    location_quantiles = {}
    for loc in sorted(locations):
        forecast_loc = forecast_df[forecast_df['location'] == loc].copy()
        if not forecast_loc.empty:
            quantile_df = forecast_loc[['target_end_date', 'output_type_id', 'value']].copy()
            quantile_df.columns = ['date', 'quantile', 'value']
            quantile_df['quantile'] = quantile_df['quantile'].astype(float)
            location_quantiles[fips_to_name.get(loc, loc)] = quantile_df
    
    all_location_quantiles.append(location_quantiles)
    forecast_labels.append(label)
    forecast_colors.append(color)

# Prepare surveillance data for each location (full data, will filter later)
location_surveillance_full = {}
for loc in sorted(locations):
    surv_loc = surv_df[surv_df['location'] == loc].copy()
    if not surv_loc.empty:
        surv_formatted = surv_loc[['date', 'value']].copy()
        surv_formatted['date'] = pd.to_datetime(surv_formatted['date'])
        location_surveillance_full[fips_to_name.get(loc, loc)] = surv_formatted


# ============================================================================
# Helper function to filter surveillance data
# ============================================================================

def filter_surveillance(location_surveillance_full, all_location_quantiles, weeks_before):
    """Filter surveillance data to specified weeks before forecast."""
    location_surveillance = {}
    for loc in location_surveillance_full:
        # Get date range across all forecasts
        all_forecast_dates = []
        for loc_quantiles in all_location_quantiles:
            if loc in loc_quantiles:
                all_forecast_dates.extend(pd.to_datetime(loc_quantiles[loc]['date']).tolist())
        
        if all_forecast_dates:
            min_forecast_date = min(all_forecast_dates)
            max_forecast_date = max(all_forecast_dates)
            
            # Start specified weeks before the forecast period
            start_date = min_forecast_date - timedelta(weeks=weeks_before)
            
            df = location_surveillance_full[loc].copy()
            location_surveillance[loc] = df[
                (df['date'] >= start_date) & (df['date'] <= max_forecast_date)
            ]
    return location_surveillance

# Create filtered surveillance for both panels
location_surveillance_left = filter_surveillance(
    location_surveillance_full, all_location_quantiles, WEEKS_BEFORE_FORECAST_LEFT
)
location_surveillance_right = filter_surveillance(
    location_surveillance_full, all_location_quantiles, WEEKS_BEFORE_FORECAST_RIGHT
)


# ============================================================================
# Create side-by-side plot
# ============================================================================

# Get sorted list of locations that have data
plot_locations = sorted([loc for loc in location_surveillance_full.keys() 
                         if loc in all_location_quantiles[0]])

n_locations = len(plot_locations)
n_cols = 4  # 4 panels per row = 2 states per row (each state gets 2 panels)
n_rows = int(np.ceil(n_locations * 2 / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3 * n_rows), squeeze=False)

# Flatten axes for easier indexing
axes_flat = axes.flatten()

for i, loc in enumerate(plot_locations):
    # Left panel index (more historical data)
    left_idx = i * 2
    # Right panel index (less historical data)
    right_idx = i * 2 + 1
    
    if left_idx >= len(axes_flat) or right_idx >= len(axes_flat):
        break
    
    ax_left = axes_flat[left_idx]
    ax_right = axes_flat[right_idx]
    
    # Plot both panels
    for ax, surv_data, weeks_label in [
        (ax_left, location_surveillance_left, f'{WEEKS_BEFORE_FORECAST_LEFT} weeks'),
        (ax_right, location_surveillance_right, f'{WEEKS_BEFORE_FORECAST_RIGHT} weeks')
    ]:
        # Add dashed grey gridlines
        ax.grid(True, linestyle='--', color='grey', alpha=0.5)
        
        # Plot surveillance data
        if loc in surv_data:
            surv = surv_data[loc]
            ax.scatter(surv['date'], surv['value'], s=10, color='black', zorder=5, label='Observed')
        
        # Plot forecasts
        for loc_quantiles, label, color in zip(all_location_quantiles, forecast_labels, forecast_colors):
            if loc in loc_quantiles:
                q_df = loc_quantiles[loc].copy()
                q_df['date'] = pd.to_datetime(q_df['date'])
                
                # Pivot to get quantiles as columns
                q_pivot = q_df.pivot(index='date', columns='quantile', values='value')
                
                # Plot median with points
                if 0.5 in q_pivot.columns:
                    ax.plot(q_pivot.index, q_pivot[0.5], color=color, linewidth=1.5, label=f'{label} (median)')
                    ax.scatter(q_pivot.index, q_pivot[0.5], color=color, s=15, zorder=4)
                
                # Plot prediction intervals (if available) - with legend labels
                intervals = [(0.025, 0.975, 0.2, '95% CrI'), (0.25, 0.75, 0.4, '50% CrI')]
                for lower, upper, alpha, interval_label in intervals:
                    if lower in q_pivot.columns and upper in q_pivot.columns:
                        ax.fill_between(q_pivot.index, q_pivot[lower], q_pivot[upper], 
                                       color=color, alpha=alpha, label=f'{interval_label}')
        
        ax.set_title(f'{loc}', fontsize=9)
        ax.tick_params(axis='x', rotation=45, labelsize=8)
        ax.tick_params(axis='y', labelsize=8)

# Hide any unused axes
for idx in range(n_locations * 2, len(axes_flat)):
    axes_flat[idx].set_visible(False)

# Add legend to first axis
axes_flat[0].legend(loc='upper left', fontsize=7)

fig.suptitle('Flu Prop ED Visits: Forecasts vs Surveillance', 
             fontsize=14, y=1.00)
fig.supylabel('Proportion ED Visits', fontsize=11)

plt.tight_layout()
plt.savefig('flu_prop_ed_smc_wmape_augmented_sidebyside.pdf', dpi=150, bbox_inches='tight')
plt.close()

Loading smc_wmape_augmented from pipeline_flu_202551_smc_wmape_202544-202550_augmented_20251217-191414-43dc0c2d_outputs_20251217-203202_output_hub_formatted.csv.gz...
